In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df=pd.read_csv('cleaned_MH_OC.csv')

In [5]:
embed_df = pd.read_csv("alphaearth_embeddings.csv")
df_embed = df.merge(embed_df,on='fid',how='left')

In [6]:
# target
target = "OC_group_median"

# remove leakage columns
leakage_cols = [
    'OC','OC_log1p','OC_bin',
    'duplicate_OC_conflict_class'
]

metadata_cols = [
    'fid','date','tile_id','tile_date',
    'source_csv','source_s2_chunk',
    'bare_soil_reason','S2_bare_soil_filter_reason',
    'spatial_qc_reason','village'
]

soil_property_cols = [
    'sand',
    'silt',
    'clay',
    'pH',
    'EC',
    'BD',
    'N',
    'P',
    'K',
    'SOC',
    'OC',
    'OC_group',
    'OC_group_median',
    'CEC',
    'moisture',
    'bulk_density'
]

drop_cols = (
    leakage_cols +
    metadata_cols +
    soil_property_cols +
    [target]
)

# keep only usable columns
X = df_embed.drop(columns=drop_cols, errors="ignore")
y = df_embed[target]

print(X.shape)

(31250, 206)


In [7]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)

X = X.fillna(X.median(numeric_only=True))

print(X.shape)

(31250, 206)


/var/folders/lm/z783vpwd6zjfsgvb0mdpm4b80000gn/T/ipykernel_3725/2924836944.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [8]:
embedding_cols = [
    c for c in embed_df.columns
    if c != "fid"
]

print("No. embeddings:", len(embedding_cols))

No. embeddings: 64


In [9]:
X_embed = X[embedding_cols]

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X_embed,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("R2:", r2_score(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

R2: 0.5041414896988032
RMSE: 0.2924921408836752


In [10]:
X_all = X.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Combined Features R2:", r2_score(y_test, pred))
print("Combined Features RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

Combined Features R2: 0.6689183948218392
Combined Features RMSE: 0.23900271899720052


## Spatial CV

In [12]:
# AlphaEarth embedding columns
embedding_cols = [c for c in embed_df.columns if c != 'fid']

# Alpha embeddings only
X_embed = X[embedding_cols]

# Combined features
X_combined = X.copy()

print("Embedding shape:", X_embed.shape)
print("Combined shape:", X_combined.shape)

Embedding shape: (31250, 64)
Combined shape: (31250, 206)


In [13]:
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold

coords = df_embed[['latitude','longitude']]

n_groups = 5

groups = KMeans(
    n_clusters=n_groups,
    random_state=42,
    n_init=10
).fit_predict(coords)

In [14]:
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

def spatial_cv(X_used, y, groups):

    gkf = GroupKFold(n_splits=5)

    r2_scores = []
    rmse_scores = []

    fold = 1

    for train_idx, test_idx in gkf.split(
        X_used,
        y,
        groups
    ):

        X_train = X_used.iloc[train_idx]
        X_test = X_used.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = RandomForestRegressor(
            n_estimators=500,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(X_test)

        r2 = r2_score(
            y_test,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )

        r2_scores.append(r2)
        rmse_scores.append(rmse)

        print(
            f"Fold {fold}: "
            f"R2={r2:.3f}, "
            f"RMSE={rmse:.3f}"
        )

        fold += 1

    return {
        "mean_r2": np.mean(r2_scores),
        "std_r2": np.std(r2_scores),
        "mean_rmse": np.mean(rmse_scores),
        "std_rmse": np.std(rmse_scores)
    }

On alphaearth only

In [15]:
embed_results = spatial_cv(
    X_embed,
    y,
    groups
)

print("\nAlphaEarth Results")
print(embed_results)

Fold 1: R2=0.340, RMSE=0.491
Fold 2: R2=0.226, RMSE=0.379
Fold 3: R2=-0.680, RMSE=0.374
Fold 4: R2=-0.132, RMSE=0.265
Fold 5: R2=-0.132, RMSE=0.248

AlphaEarth Results
{'mean_r2': np.float64(-0.07567616917668436), 'std_r2': np.float64(0.35667968757503227), 'mean_rmse': np.float64(0.35147423745672896), 'std_rmse': np.float64(0.08826493252357011)}


Combined 

In [16]:
combined_results = spatial_cv(
    X_combined,
    y,
    groups
)

print("\nCombined Results")
print(combined_results)

Fold 1: R2=0.462, RMSE=0.444
Fold 2: R2=0.294, RMSE=0.362
Fold 3: R2=-0.168, RMSE=0.312
Fold 4: R2=0.232, RMSE=0.218
Fold 5: R2=-1.644, RMSE=0.379

Combined Results
{'mean_r2': np.float64(-0.16481501366907653), 'std_r2': np.float64(0.7679259117550804), 'mean_rmse': np.float64(0.34297960419131046), 'std_rmse': np.float64(0.0752670729542772)}
